Intro:
Notebook ini adalah Full System Scheduler yang mana menjadi main system untuk mengCreate event/jadwal User ke dalam Google Calendar dengan menggunakan input/query tertentu dari user.
Bagian Notebook system Scheduler ini akan dibagi kedalam 5 Part Notes:
1. **Get Audio Transcription**: Menyediakan cara-cara scheduler mendapatkan input/query dari user mengenai aktivitas yang hendak dimasukkan dalam kalender. Terdapat 3 cara user memberi input mengenai aktivitas yang hendak dijadwalkan yakni: record audio, import audio, ataupun langsung write transcript text manual.
2. **Getting the Schedule-able information with spaCy**: Menggunakan spaCy untuk mengubah input/query user ke dalam pecahan data yang dibutuhkan (date, time, activity)
3. **Parsing the Date and time**: di bagian ini, informasi date dan time akan diparsing ke dalam bentuk yang mana dapat digunakan untuk create event dalam google calendar -> string, datetime.date(), datetime.time()
4. **Get the activity label**: Di bagian ini model BERT yang telah dilatih sebelumnya akan diload dan digunakan untuk mengklasifikasikan activity user yang nantinya akan dimasukkan ke dalam bagian 'description' dalam google calendar
5. **Input to Google Calendar API**: bagian terakhir dari sistem Scheduler di mana informasi-informasi input/query dari user akan dibentuk sebagai suatu Event baru di Google Calendar user

***!!! Notes***: Sebelum menjalankan code ini, baik untuk mengupload terlebih dahulu file:
  

1.   YahooTopic_test.csv,
2.   last_trained_model_checkpoint_1.0.pth,
3.   inlaid-sentinel-415410-75127567c3f6.json,
4.   client_secret_996989308703-papulo6s1jt8231kf2vrr1kiml0jqevu.apps.googleusercontent.com.json

dari Drive dalam Notion: https://tsukishimaalan20.notion.site/NLP-AOL-Language-based-Scheduler-b0a63589a4b84187b8316cf0450d2c9c?pvs=4

**Penjelasan lebih detail terkait Whisper sebagai STT juga dijabarkan dalam halaman notion tersebut.**

#  1. Get Audio Transcription

Pada bagian 'Get Audio Transcription' terdapat 3 cara yang bisa dipilih user untuk menginput informasi:
1. Plan A: Record Audio
2. Plan B: Import Audio
3. Plan C: Input ketik langsung query

Pada bagian ini digunakan Library IO untuk membantu proses record Audio serta Whisper yang digunakan untuk proses transcription Speech to Text (STT)

## Plan A = IO + Whisper

### Record Audio: IO

In [ ]:
# Installation library2 yang tidak ada di Google Colab
!pip install sounddevice wavio openai-whisper ffmpeg-python ipywidgets noisereduce scipy numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.6/798.6 kB 14.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 49.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 68.9 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvid

In [ ]:
from IPython.display import display, Javascript
from google.colab import output
import io
import base64
import soundfile as sf
import numpy as np

# Berikut adalah JS yang dipakai untuk merecord audio dan menyimpannya sebagai recording.wav
# Terdapat pula button untuk start & stop recording, serta button save untuk memastikan file saved
RECORD_AUDIO_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));

var record = true;
var mediaRecorder;
var audioChunks = [];

const startRecording = async () => {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  mediaRecorder = new MediaRecorder(stream);
  audioChunks = [];
  mediaRecorder.addEventListener('dataavailable', event => {
    audioChunks.push(event.data);
  });

  mediaRecorder.addEventListener('stop', async () => {
    const audioBlob = new Blob(audioChunks);
    const arrayBuffer = await audioBlob.arrayBuffer();
    const audioBuffer = new Uint8Array(arrayBuffer);
    const base64String = btoa(String.fromCharCode(...audioBuffer));
    google.colab.kernel.invokeFunction('notebook.record_audio', [base64String], {});
    document.getElementById("save_button").disabled = false;
  });

  mediaRecorder.start();
  while (record) {
    await sleep(100);
  }
  mediaRecorder.stop();
}

const saveRecording = async () => {
  record = false;
  mediaRecorder.stop();
}

if (document.querySelector("#record_button") == null) {
  const button = document.createElement('button');
  button.id = "record_button";
  button.textContent = "Start Recording";
  button.onclick = () => {
    if (record) {
      record = false;
      button.textContent = "Start Recording";
    } else {
      record = true;
      startRecording();
      button.textContent = "Stop Recording";
    }
  };
  document.body.appendChild(button);

  const saveButton = document.createElement('button');
  saveButton.id = "save_button";
  saveButton.textContent = "Save Recording";
  saveButton.disabled = true;
  saveButton.onclick = saveRecording;
  document.body.appendChild(saveButton);
}
"""

# Berikut adalah main function untuk proses record audio
def record_audio(base64_string):
    audio_bytes = io.BytesIO(base64.b64decode(base64_string)) #Decoding the input (base64 string)-create BytesIO
    with open("recording.wav", "wb") as f:
        f.write(audio_bytes.getbuffer())

output.register_callback('notebook.record_audio', record_audio) # Register the function as callback -> Agar bisa berjalan dalam JS

display(Javascript(RECORD_AUDIO_JS)) # Jalankan proses recording by JS

<IPython.core.display.Javascript object>

### STT: whisper

In [ ]:
import whisper
import noisereduce as nr
import scipy.io.wavfile as wav
import numpy as np
import time

# Fungsi untuk mengurangi noise pada file audio
def reduce_noise(input_file, output_file):
    # Membaca file audio
    rate, data = wav.read(input_file)

    # Mengurangi noise menggunakan noisereduce
    reduced_noise = nr.reduce_noise(y=data, sr=rate)

    # Menyimpan hasil noise reduction ke file baru
    wav.write(output_file, rate, reduced_noise)

# Fungsi untuk mentranskripsi file audio menggunakan Whisper
def transcribe_audio(filename):
    model = whisper.load_model("base")  # Load base model dari library Whisper
    result = model.transcribe(filename, language='en')  # Transkrip dengan bahasa Inggris
    text = result["text"]  # Menyimpan hasil transkrip
    print("Transcription: " + text)  # Menampilkan hasil transkrip
    return text

# Nama file sebelum dan sesudah noise reduction
filename = "recording.wav"       # Nama file asli
clean_filename = "cleaned.wav"   # Nama file setelah noise reduction

# Proses noise reduction
reduce_noise(filename, clean_filename)

# Memulai proses transkripsi dengan file yang telah dibersihkan
transcription = transcribe_audio(clean_filename)

time.sleep(5)  # Jeda untuk memastikan proses selesai


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 106MiB/s]
/usr/local/lib/python3.10/dist-packages/whisper/transcribe.py:115: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcription:  I want to go to the library at 7b and next Sunday.


## Plan B: Import Audio + Whisper

In [ ]:
# Menginstall library yang belum terinstall dalam Google Colab

!pip install sounddevice wavio openai-whisper ffmpeg-python
!sudo apt update
!sudo apt install ffmpeg

# sounddevice wavio openai-whisper ffmpeg-python ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.6/798.6 kB 5.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 8.2 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cac

In [ ]:
from google.colab import files
uploaded = files.upload() # untuk proses uploading/importing audio file

In [ ]:
import whisper

# Proses transcription Whisper yang sama seperti telah dijelaskan di Plan A
def transcribe_audio(filename):
    model = whisper.load_model("base")
    result = model.transcribe(filename)
    text = result["text"]
    print("Transcription: " + text)
    return text

# Memulai proses transkripsi untuk file audio yang diupload
filename = list(uploaded.keys())[0]
transcription = transcribe_audio(filename)


## Plan C: Override Transcription

In [ ]:
# TEMP OVERRIDE
# Tulis langsung mengenai Query/Input dari user

# Contoh
transcription = "Learn Taekwondo tomorrow at 7 pm"
transcription = "Go to the Gym tomorrow at 7 pm"
transcription = "Play golf in Jakarta in 5 days at 7 pm"

# Masukkan informasi di sini
transcription = input("Type your activity here in English: ")

Type your activity here in English: Go to the Library tomorrow at 7 pm


# 2. Getting the Schedule-able information with spaCy

In [ ]:
# Download transformer pretrained model dari spacy
!python -m spacy download en_core_web_trf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.3/236.3 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.6/731.6 kB 24.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# Import library spacy, Matcher, dan load transformer Model
import spacy
from spacy.matcher import Matcher
nlp = spacy.load("en_core_web_trf")

In [ ]:
# Function untuk mengekstrak informasi yang dibutuhkan untuk membuat jadwal dalam kalender
# Activity, Date dan Time
def extract_activity_and_time(doc):
    activities = []
    times = []
    date = []

    # Mengekstrak entitas yang berlabel TIME sebagai waktu jadwal
    for ent in doc.ents:
        if ent.label_ == "TIME":
            times.append(ent.text)

    # Mengekstrak entitas yang berlabel DATE sebagai tanggal jadwal
    for ent in doc.ents:
        if ent.label_ == "DATE":
            date.append(ent.text)

    matcher = Matcher(nlp.vocab)

    # Create pattern untuk kemungkinan membuat statement activity
    pattern = [
               {"POS": {"IN": ["VERB", "NOUN"]}},
               {"POS": "ADP", "OP": "?"},
               {"POS": "DET", "OP": "?"},
               {"POS": "NOUN", "OP": "?"},
               {"POS": "ADP", "OP": "?"},
               {"POS": "PROPN", "OP": "?"}
               ]

    # pattern = [
        # {"LEMMA": "schedule", "POS": "VERB", "OP": "?"},  # Optional verb 'schedule'
        # {"POS": "VERB"},  # Optional verb 'schedule'
        # {"POS": "DET", "OP": "?"},  # Optional determiner
        # {"POS": "ADJ", "OP": "*"},  # Any number of adjectives
        # {"POS": "NOUN"},            # Mandatory noun
        # {"POS": "ADP", "OP": "?"},  # Optional preposition (like 'on', 'at')
        # {"POS": "DET", "OP": "?"},  # Optional determiner (before date/time)
        # {"ENT_TYPE": "DATE", "OP": "?"},  # Optional date entity
        # {"ENT_TYPE": "TIME", "OP": "?"}   # Optional time entity
        # {"POS": "DET", "OP": "?"},  # optional: match 0 or 1 times
        # {"POS": "NOUN"}
    # ]

    # Menambahkan pattern ke dalam matcher
    matcher.add("ACTIVITY_PATTERN", [pattern])

    # Memulai proses pengenalan pattern dalam doc
    matches = matcher(doc)

    # Spanning text doc untuk kemungkinan statemen ke dalam activities list
    for match_id, start, end in matches:
        span = doc[start:end]
        activities.append(span.text)

    return activities, times, date

In [ ]:
def data_check_calendar(activity, time, date):
    print("Activity:", activity)
    print("Time:", time)
    print("Date: ", date)
    print("Information checking complete...")

In [ ]:
# Block Code untuk display data-data input sebagai proses checking

# mengalihkan transcription hanya untuk proses checking
sentence = transcription

# Printing the text dan tag untuk transkripsi yang telah dimasukkan ke dalam model
for token in nlp(sentence):
    print(f'{token.text:15} {token.pos_}')
print('------------------------------------------')

# Menjalankan fungsi ekstraksi data-data
activity, time, date = extract_activity_and_time(nlp(sentence))

# Printing data-data input user yang telah terekstrak
if activity or time:
    data_check_calendar(activity, time, date)
else:
    print("Activity or time not found in the transcription.")
print("+++++++++++++++++++++++++++++++++++++++++++++++++++++++++")

Go              VERB
to              ADP
the             DET
Library         NOUN
tomorrow        NOUN
at              ADP
7               NUM
pm              NOUN
------------------------------------------
Activity: ['Go', 'Go to', 'Go to the', 'Library', 'Go to the Library', 'tomorrow', 'Library tomorrow', 'tomorrow at', 'Library tomorrow at', 'pm']
Time: ['7 pm']
Date:  ['tomorrow']
Information checking complete...
+++++++++++++++++++++++++++++++++++++++++++++++++++++++++


In [ ]:
# Block code ini menjalankan proses eliminasi terhadap kemungkinan statement activity

# Menyimpan setiap kata yang terlabel sebagai Date dan Time
words = []
for sentence in (date+time):
  for word in sentence.split(" "):
    words.append(word)

# Function untuk cek apakah suatu kemungkinan statement mengandung kata dalam Date dan Time
def contains_word_from_set(sentence, word_set):
    return any(word in word_set for word in sentence.split())

# Eliminasi kemungkinan statement yang mengandung label Date and Time dari sentence
filtered_ActivityList = [sentence for sentence in activity if not contains_word_from_set(sentence, words)]

print(filtered_ActivityList)


['Go', 'Go to', 'Go to the', 'Library', 'Go to the Library']


In [ ]:
def sort_string_list_by_length(string_list):
    # Sort kemungkinan2 activity statement based on the length of setiap string
    sorted_list = sorted(string_list, key=len, reverse=True)
    return sorted_list

In [ ]:
# Menjalankan proses sorting
filtered_ActivityList = sort_string_list_by_length(filtered_ActivityList)

# Menampilkan kembali data-data dengan statement activity memakai string terpanjang=terlengkap
filtered_ActivityList[0], date, time

('Go to the Library', ['tomorrow'], ['7 pm'])

# 3. Parsing the Date and Time

In [ ]:
from dateutil import parser
from datetime import datetime, timedelta
import re

# Block Code ini berisi function-function untuk proses parsing date ke dalam bentuk yang dapat dipakai dalam create Event Google Calendar
# -> bentuk data type: datetime.date()

# Function untuk parsing tanggal dengan input weekdays
def parse_relative_weekday(weekday_str):
    today = datetime.now()
    weekdays = {
        'monday': 0,
        'tuesday': 1,
        'wednesday': 2,
        'thursday': 3,
        'friday': 4,
        'saturday': 5,
        'sunday': 6
    }
    weekday = weekdays[weekday_str.lower()]
    days_ahead = weekday - today.weekday()
    if days_ahead <= 0:
        days_ahead += 7
    return (today + timedelta(days=days_ahead)).date()

# Function untuk parsing tanggal dengan input ordinal weekdays
def parse_ordinal_weekday(expression):
    today = datetime.now()
    match = re.match(r'the (\d+)(?:st|nd|rd|th) (\w+) of next (\w+)', expression, re.IGNORECASE)
    if match:
        ordinal, weekday_str, month_str = match.groups()
        ordinal = int(ordinal)
        weekdays = {
            'monday': 0,
            'tuesday': 1,
            'wednesday': 2,
            'thursday': 3,
            'friday': 4,
            'saturday': 5,
            'sunday': 6
        }
        months = {
            'january': 1,
            'february': 2,
            'march': 3,
            'april': 4,
            'may': 5,
            'june': 6,
            'july': 7,
            'august': 8,
            'september': 9,
            'october': 10,
            'november': 11,
            'december': 12
        }
        target_month = months[month_str.lower()]
        target_year = today.year if today.month < target_month else today.year + 1
        first_day_of_month = datetime(target_year, target_month, 1)
        first_weekday_of_month = first_day_of_month + timedelta(days=(weekdays[weekday_str.lower()] - first_day_of_month.weekday() + 7) % 7)
        return (first_weekday_of_month + timedelta(weeks=ordinal - 1)).date()
    return None

# Proses parsing date dilakukan 'manual' dengan memasukkan setiap kemungkinan kondisi statement yang menginformasikan tanggal
def convert_to_date(date_str):
    try:
        # Parsing untuk absolute dates seperti 'June 23 2024'
        parsed_date = parser.parse(date_str)
        return parsed_date.date()
    except (ValueError, OverflowError):
        pass

    # Handle parsing 'next week'
    if re.match(r'next week', date_str, re.IGNORECASE):
        return (datetime.now() + timedelta(weeks=1)).date()

    # Handle parsing 'X weeks'
    match = re.match(r'(\d+) weeks', date_str, re.IGNORECASE)
    if match:
        weeks = int(match.group(1))
        return (datetime.now() + timedelta(weeks=weeks)).date()

    # Handle parsing 'next month'
    if re.match(r'next month', date_str, re.IGNORECASE):
        today = datetime.now()
        next_month = today.month + 1 if today.month < 12 else 1
        year = today.year if next_month != 1 else today.year + 1
        day = min(today.day, (datetime(year, next_month + 1, 1) - timedelta(days=1)).day)
        return datetime(year, next_month, day).date()

    # Handle parsing 'X months'
    match = re.match(r'(\d+) months', date_str, re.IGNORECASE)
    if match:
        months = int(match.group(1))
        today = datetime.now()
        month = today.month - 1 + months
        year = today.year + month // 12
        month = month % 12 + 1
        day = min(today.day, (datetime(year, month + 1, 1) - timedelta(days=1)).day)
        return datetime(year, month, day).date()

    # Handle parsing 'next year'
    if re.match(r'next year', date_str, re.IGNORECASE):
        return datetime(datetime.now().year + 1, datetime.now().month, datetime.now().day).date()

    # Handle parsing 'X years'
    match = re.match(r'(\d+) years', date_str, re.IGNORECASE)
    if match:
        years = int(match.group(1))
        today = datetime.now()
        return datetime(today.year + years, today.month, today.day).date()

    # Handle parsing relative weekdays ('Next Monday')
    match = re.match(r'Next (\w+)', date_str, re.IGNORECASE)
    if match:
        return parse_relative_weekday(match.group(1))

    # Handle parsing 'tomorrow'
    if re.match(r'tomorrow', date_str, re.IGNORECASE):
        return (datetime.now() + timedelta(days=1)).date()

    # Handle parsing 'X days'
    match = re.match(r'(\d+) days', date_str, re.IGNORECASE)
    if match:
        days = int(match.group(1))
        return (datetime.now() + timedelta(days=days)).date()

    # Handle parsing ordinal weekdays ('the first Sunday of next May')
    parsed_date = parse_ordinal_weekday(date_str)
    if parsed_date:
        return parsed_date
    return None



In [ ]:
# Function untuk proses parsing statement label TIME user ke dalam bentuk Time yang dapat diinput dalam Google Calendar (datetime.time())
def convert_to_time(time_str):
    try:
        parsed_date = parser.parse(time_str)
        return parsed_date.time()
    except (ValueError, OverflowError):
        pass

In [ ]:
# Memulai parsing date
# Bila info label DATE tidak ditemukan dalam statement -> Secara default akan diset pada tanggal user memasukkan input
if len(date) == 0:
  parsed_date = datetime.now().date()
  date.append('Today')
else: parsed_date = convert_to_date(date[0])

print(f"Original DATE: {date[0]} -> Parsed DATE: {parsed_date}")


# Memulai parsing time
# Bila info label TIME tidak ditemukan dalam statement -> Secara default akan diset pada waktu user memasukkan input
if len(time) == 0:
  time.append('Now time')
  parsed_time = datetime.now().time()
else: parsed_time = convert_to_time(time[0])
print(f"Original TIME: {time[0]} -> Parsed TIME: {parsed_time}")

Original DATE: tomorrow -> Parsed DATE: 2024-06-27
Original TIME: 7 pm -> Parsed TIME: 19:00:00


In [ ]:
# Checking kelengkapan informasi pembuatan Event calendar
filtered_ActivityList[0], parsed_date, parsed_time

('Go to the Library', datetime.date(2024, 6, 27), datetime.time(19, 0))

In [ ]:
# Checking data type untuk setiap informasi
type(filtered_ActivityList[0]), type(parsed_date), type(parsed_time)

(str, datetime.date, datetime.time)

# 4. Get the Activity Label

Bagian ini merupakan proses mengklasifikan topik/tipe kegiatan dari activity yang diinput user untuk dijadwalkan.
Proses klasifikasi akan menggunakan model BERT yang telah di Fine-Tuned sebelumnya pada notebook training BERT model (BERT_NLP_AOL.ipynb)->{[link text](https://colab.research.google.com/drive/1tO55YMBdIFISk9veBg2HLeN5vgIhXtIh?usp=sharing)}


In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertTokenizer, BertForSequenceClassification
import pandas as pd

# Import kembali dataset yang digunakan untuk proses training
dataset = pd.read_csv('/content/YahooTopic_test.csv')
dataset.columns = ['label', 'title', 'content', 'answer']
dataset = dataset.fillna(" ")

# Initialize BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Load pre-trained BERT model for sequence classification sebagai initialize base model
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=dataset['label'].nunique())

# Declare parameter yang diperlukan
learning_rate = 2e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
loss_fn = torch.nn.CrossEntropyLoss()

# Set model ke device yang ada (CPU/GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Load checkpoint hasil save dari proses training BERT model sebelumnya
checkpoint = torch.load('/content/last_trained_model_checkpoint_BERT_AOLSR.pth' , map_location=torch.device('cpu'))

# Load Fine Tuned Bert model and optimizer states
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Ambil bagian label dataset
topic_labels = dataset['label']

# Instantiate LabelEncoder
label_encoder = LabelEncoder()

# Fit label encoder pada label dataset
label_encoder.fit(topic_labels)


In [ ]:
# Function untuk decode kembali label ke dalam tipe aktivitas nya
def decoding(num):
  if num == 1: return 'social'
  elif num == 2: return 'education'
  elif num == 3: return 'health'
  elif num == 4: return 'career'
  else: return 'hobby'

In [ ]:
def predict_topic(text, label_encoder):

    # Tokenize text input menggunakan tokenizeer yang sama yang dipakai dalam training (BERT Tokenizer)
    # Tokenize dengan setting yang ditentukan berikut dan return dalam bentuk pytorch
    inputs = tokenizer.encode_plus(text, add_special_tokens=True, max_length=128, padding='max_length', truncation=True, return_tensors='pt')

    # Ambil bagian vector id tensor dan attention mask; pindahkan dalam device yang sesuai
    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)

    # Menjalankan proses prediksi
    with torch.no_grad():
        model.eval() #atur model ke mode evaluasi->nonaktivasi mode tertentu yang aktif kalau mode training
        outputs = model(input_ids, attention_mask=attention_mask)
    logits = outputs.logits #ambil value hasil dalam bentuk logits

    # Ambil predicted label dengan indeks nilai logit tertinggi dan ubah jadi bilangan bulat
    predicted_label = torch.argmax(logits, dim=1).item()

    # Decode the predicted label using label_encoder
    predicted_topic = label_encoder.inverse_transform([predicted_label])[0]

    return decoding(predicted_topic), logits

# Mengaktifkan fungsi prediksi klasifikasi topik aktifitas
topicPredicted = predict_topic(filtered_ActivityList[0], label_encoder)

# 5. Input to GoogleCalendar API

Block code bagian terakhir ini berfungsi untuk create event baru user ke dalam Google Calendar

*This project integrates the Google Calendar API using OAuth 2.0 and service account authentication to manage event scheduling. The service account credentials and client secrets are securely handled and not included in this public document. The code demonstrates how events are dynamically created in Google Calendar based on user input.*

In [ ]:
# Mengimpor library yang diperlukan untuk autentikasi dan akses API Google.
from google.oauth2 import service_account
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

SERVICE_ACCOUNT_FILE = 'path/to/your-service-account-file.json'
SCOPES = ['https://www.googleapis.com/auth/calendar'] # Scope izin untuk Google Calendar


# Memuat credentials dari Google Service
credentials = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)

# Build service API Google Calendar berdasar credentials tadi
service = build('calendar', 'v3', credentials=credentials)

# Proses Autentikasi OAuth 2.0
CLIENT_SECRET_FILE = 'path/to/your-client-secret-file.json'
SCOPES = ['https://www.googleapis.com/auth/calendar']
flow = InstalledAppFlow.from_client_secrets_file(CLIENT_SECRET_FILE, scopes=SCOPES) #Inisialisasi alur autentikasi
credentials = flow.run_local_server(port=0) #Jalankan proses autentikasi dengan server local
service = build('calendar', 'v3', credentials=credentials)

# Build function untuk create Event dari informasi yang ada untuk dijadwalkan dalam Google Calendar
def create_event(service, summary, start_time, end_time, description, location=''):
  event = {
      'summary': summary, #-> Statement Activity
      'location': location, #-> Location activity (pada sistem ini masih dikosongkan)
      'description': description, #-> Bagian Description akan diisi dengan label topic prediction

      # -> Pada bagian start & end datetime akan disamakan
      'start': {
          'dateTime': start_time,
          'timeZone': 'Asia/Jakarta',
      },
      'end': {
          'dateTime': end_time,
          'timeZone': 'Asia/Jakarta',
      }
  }

  # Insert event dalam Google calendar
  event = service.events().insert(calendarId='primary', body=event).execute()

  # Display link untuk checking agenda yang telah dibuat
  print('Event created: %s' % (event.get('htmlLink')))

def main():

  # Masukkan informasi date dan time user sebagai start sekaligus end time
  start_datetime = f"{parsed_date}T{parsed_time}:00"
  end_datetime = f"{parsed_date}T{parsed_time}:00"

  # Menjalankan proses create event dan insert dalam google calendar
  create_event(service, filtered_ActivityList[0], start_datetime, end_datetime, description=topicPredicted)

if __name__ == "__main__":
    main() #Menjalankan main function